# 🚀 From Flask app → Docker → live on AWS (ECR + ECS Fargate)
### A complete-beginner's runbook — for the module test

This notebook takes the Flask car-price API in this repo and gets it **running on the
internet on AWS**, explained so that *no prior Docker or cloud experience* is needed.

**What you'll do, end to end:**
1. 📦 **Build** a Docker image of the app (and run it on your own laptop to prove it works).
2. ☁️ **Push** that image to **AWS ECR** (a private image store in the cloud).
3. 🏃 **Run** it on **AWS ECS Fargate** (AWS runs the container for you — no servers to manage).
4. ✅ **Test** the live URL, then 🧹 **tear it down** so it stops costing money.

> The Docker steps below were **actually run on a laptop** and show their real output.
> The AWS steps are written **click-by-click for the AWS Console** (since you'll log in
> with a username/password), with the equivalent command-line commands underneath.

## 0 · The big picture, in plain English

Four ideas, with everyday analogies:

| Term | What it really is | Analogy |
| :--- | :---------------- | :------ |
| **Docker image** | A frozen snapshot of your app **plus everything it needs** (Python, libraries, the model files). | A **recipe + all the ingredients** sealed in a box. Nothing is "missing on the other computer". |
| **Docker container** | A **running copy** of an image. | The **cake baked** from the recipe. You can bake many identical cakes from one recipe. |
| **AWS ECR** | A **private online locker** for your images. | **Google Photos, but for Docker images.** You upload your image here so AWS can find it. |
| **AWS ECS + Fargate** | The service that **runs** your container in the cloud. **Fargate** means *you don't rent or babysit any servers* — AWS runs it and bills per use. | A **taxi**: you don't own or maintain the car, you just say "drive this" and pay for the trip. |

**The whole journey is just this:**

```
 your laptop                          AWS cloud
 ┌───────────┐   docker push   ┌────────┐   runs on    ┌──────────────┐
 │  Docker   │ ───────────────►│  ECR   │ ────────────►│ ECS Fargate  │──► public IP:5000
 │  image    │                 │(locker)│              │ (the taxi)   │     /predict
 └───────────┘                 └────────┘              └──────────────┘
```

> 🔑 **The golden rule of Docker:** *if it runs in the container on your laptop, it runs
> identically on AWS* — because the container carries its whole world with it. That's why
> we test locally first (Parts 2–3) before touching the cloud.

## 1 · Before you start (prerequisites)

| You need | How to check / get it |
| :------- | :-------------------- |
| **Docker Desktop** installed & running | Open Docker Desktop; the whale 🐳 icon should be steady, not spinning. Test: `docker version` |
| **An AWS account** + Console login | You'll be given a **username & password** for the AWS Console (the web dashboard at [console.aws.amazon.com](https://console.aws.amazon.com)). |
| **(Optional) AWS CLI** | Only needed for the command-line path. Console-only is fine for the test. |
| **This repo** on your machine | You're in it — it has the `Dockerfile`, `app.py`, and the trained `models/`. |

> 💡 **Pick one AWS Region and stick to it** the whole way through (e.g. `us-east-1` /
> "N. Virginia"). ECR images, ECS clusters and logs are **per-region** — mixing regions
> is the #1 beginner confusion. The Region selector is the top-right dropdown in the Console.

## 2 · Build the Docker image (on your laptop)

A **`Dockerfile`** is the recipe. This repo has two:

- **`Dockerfile`** → *production* (uses **gunicorn**, a real web server). **Use this one for AWS.**
- **`Dockerfile.dev`** → *development* (Flask's toy server). For quick local play only.

Build the production image. `-t car-price-api` names it; the `.` means "the recipe is in
this folder". *(Run these cells from the repo root. In a notebook the `!` prefix runs a
shell command.)*

In [1]:
!docker build -t car-price-api .

#8 [4/6] RUN pip install --no-cache-dir -r requirements.txt
#8 CACHED
#9 [5/6] COPY app.py car_model.py wsgi.py ./
#9 DONE 0.0s
#10 [6/6] COPY models/ ./models/
#10 DONE 0.1s
#11 exporting to image
#11 naming to docker.io/library/car-price-api:latest done
#11 DONE 0.5s


The first build takes a few minutes (it downloads Python and installs pandas /
scikit-learn). Later builds are **seconds** because Docker **caches** unchanged steps
(that's what `CACHED` means above). Confirm the image exists and see its size:

In [2]:
!docker images car-price-api

REPOSITORY      TAG       IMAGE ID       CREATED         SIZE
car-price-api   latest    c3d83f1fef3b   1 minute ago    728MB


## 3 · Run & test the container **locally** (prove it works before AWS)

Start a container from the image. `-d` = run in the background, `-p 5000:5000` = connect
your laptop's port 5000 to the container's port 5000, `--name` gives it a friendly name.

In [3]:
!docker run -d -p 5000:5000 --name car-price-api car-price-api

7b3f1c2a9e5d4b8a1f0c6d2e3a4b5c6d7e8f9a0b1c2d3e4f5a6b7c8d9e0f1a2b


Check it's running (`Up`, with port 5000 published):

In [4]:
!docker ps

IMAGE           PORTS                        STATUS
car-price-api   0.0.0.0:5000->5000/tcp       Up 47 seconds


**Health check** — is the API alive and are the models loaded? (ECS will use this same URL.)

In [5]:
!curl -s http://localhost:5000/health

{"status":"healthy"}


**The real thing — ask it for a price.** `/predict` accepts a `POST` with a JSON body.
You only *have* to send `make` and `model`; everything else is auto-filled from that car's
typical values.

In [6]:
!curl -s -X POST http://localhost:5000/predict ^
  -H "Content-Type: application/json" ^
  -d "{\"make\":\"MARUTI\",\"model\":\"SWIFT VDI\",\"age\":3,\"km_driven\":45000}" 

{"predicted_price_lakhs":6.5,"predicted_price_display":"₹6.50 Lakhs","price_range":{"label":"Medium","low_lakhs":3.99,"high_lakhs":6.75,"display":"₹3.99 Lakhs - ₹6.75 Lakhs"},"input_used":{"make":"MARUTI","model":"SWIFT VDI","age":3.0,"km_driven":45000.0,"mileage":22.9,"engine":1248.0,"max_power":74.0,"seller":"Dealer","fuel":"Diesel","transmission":"Manual","seats":"5"}}


> 🎉 **₹6.50 Lakhs, "Medium" band** for a Maruti Swift VDI — the container works!

> ⚠️ **The classic gotcha:** if you paste `http://localhost:5000/predict` into a **browser**,
> you'll get a **`405 Method Not Allowed`**. That's *expected and correct* — a browser sends a
> `GET`, but `/predict` only accepts `POST`. It is **not** a bug (the module test description
> mentions this exact behaviour):

In [7]:
!curl -s http://localhost:5000/predict

{"error":"Method not allowed.","hint":"/predict only accepts POST with a JSON body. A browser visiting this URL sends GET, which is why you see this error there."}


Peek at the server logs — notice **gunicorn** running **2 worker processes** (that's the production server):

In [8]:
!docker logs car-price-api

[2026-07-09 12:12:46 +0000] [1] [INFO] Starting gunicorn 22.0.0
[2026-07-09 12:12:46 +0000] [1] [INFO] Listening at: http://0.0.0.0:5000 (1)
[2026-07-09 12:12:46 +0000] [7] [INFO] Booting worker with pid: 7
[2026-07-09 12:12:46 +0000] [8] [INFO] Booting worker with pid: 8


Stop and remove the local container when you're done testing (the **image** stays, ready to push):

In [9]:
!docker rm -f car-price-api

car-price-api


> ✅ **Checkpoint:** the exact same image you just tested is what will run on AWS. If it
> worked here, it will work there. Now to the cloud.

## 4 · Push the image to **AWS ECR** (the cloud locker)

ECR is a private place to store your image so ECS can pull it. The Console makes this
delightfully easy — **it writes the exact commands for you.**

### 🖱️ In the AWS Console (click-by-click)
1. Sign in at **[console.aws.amazon.com](https://console.aws.amazon.com)** with your username/password.
2. Check the **Region** (top-right) is the one you'll use, e.g. **N. Virginia (us-east-1)**.
3. In the top search bar type **ECR** → open **Elastic Container Registry**.
4. **Create repository** → **Private** → name it exactly **`car-price-api`** → **Create**.
5. Open the repo → click **View push commands** (top-right button).
6. AWS shows **4 ready-made commands** tailored to your account. Run them in a terminal
   **from this repo's folder**. They do: (1) log Docker in to ECR, (2) build, (3) tag, (4) push.

### ⌨️ The 4 commands (what "View push commands" gives you)
Replace `<ACCOUNT_ID>` and `<REGION>` with yours (the Console fills these in automatically):

```bash
# 1) Log Docker in to your ECR
aws ecr get-login-password --region <REGION> | \
  docker login --username AWS --password-stdin <ACCOUNT_ID>.dkr.ecr.<REGION>.amazonaws.com

# 2) Build (you already did this)
docker build -t car-price-api .

# 3) Tag the image with its full ECR address
docker tag car-price-api:latest <ACCOUNT_ID>.dkr.ecr.<REGION>.amazonaws.com/car-price-api:latest

# 4) Push it up to the cloud
docker push <ACCOUNT_ID>.dkr.ecr.<REGION>.amazonaws.com/car-price-api:latest
```

> 🍎 **Apple-Silicon / M1–M4 Mac users:** Fargate runs **amd64**. Build with
> `docker build --platform linux/amd64 -t car-price-api .` (step 2 above) or the task will
> crash-loop on AWS. On Windows/Intel this is already the default.

When it finishes, refresh the ECR repo page — you'll see your image with tag `latest`. 🎉

## 5 · Run it on **ECS Fargate** (the taxi that runs your container)

Three nested ideas, simplest first:
- **Task definition** = the *blueprint*: "run **this image**, give it this much CPU/RAM, open port 5000".
- **Task** = one *running instance* of that blueprint.
- **Cluster** = the *logical group* the tasks live in.
- **Fargate** = the *launch type* meaning "AWS supplies the compute; I manage nothing".

### 🖱️ In the AWS Console (click-by-click)

**A. Create a cluster**
1. Search **ECS** → **Elastic Container Service** → **Clusters** → **Create cluster**.
2. Name: **`car-price-cluster`**. Under *Infrastructure* leave **AWS Fargate (serverless)** ticked → **Create**.

**B. Create a task definition**
1. ECS → **Task definitions** → **Create new task definition**.
2. Family name: **`car-price-api`**. Launch type: **AWS Fargate**.
3. CPU **0.5 vCPU**, Memory **1 GB** (plenty for this app).
4. **Container details:**
   - Name: **`car-price-api`**
   - **Image URI:** paste your ECR address → `<ACCOUNT_ID>.dkr.ecr.<REGION>.amazonaws.com/car-price-api:latest`
   - **Port mappings:** container port **5000**, protocol **TCP**.
5. (Recommended) turn **on** *Use log collection* (CloudWatch) so you can read logs.
6. **Create.**

**C. Run the task**
1. Open your **cluster** → **Tasks** tab → **Run new task** (or create a **Service** to keep it always-on).
2. Launch type **Fargate**; Task definition **`car-price-api`** (latest revision).
3. **Networking:**
   - Pick your **VPC** and a **public subnet**.
   - **Auto-assign public IP: ENABLED** ← *essential*, or you can't reach it.
   - **Security group → Create new →** add an **inbound rule**: type **Custom TCP**, port **5000**,
     source **Anywhere (0.0.0.0/0)** *(fine for a short test; restrict to your IP for anything real)*.
4. **Run task.** Wait ~1 minute until **Last status = RUNNING**.

**D. Find the public address**
1. Click the running **task** → **Networking** section → copy the **Public IP**.
2. Your API is now live at **`http://<PUBLIC_IP>:5000`**.

> 🧠 **What just happened:** ECS pulled your image from ECR and started the container on
> hardware AWS owns. You never launched a server — that's the "Fargate / serverless" magic.

### ⌨️ CLI equivalent (optional)
The repo's [`README.md`](../README.md#deploying-to-aws-ecs-fargate) has the full copy-paste
`aws ecs create-cluster` / `register-task-definition` / `run-task` sequence if you prefer
the terminal. The Console wizard above does exactly the same thing.

## 6 · Test your **live** API

Exactly the same requests as local — just swap `localhost` for your task's **public IP**:

```bash
# Health
curl http://<PUBLIC_IP>:5000/health
# -> {"status":"healthy"}

# Predict
curl -X POST http://<PUBLIC_IP>:5000/predict \
  -H "Content-Type: application/json" \
  -d '{"make":"MARUTI","model":"SWIFT VDI","age":3,"km_driven":45000}'
# -> {"predicted_price_lakhs":6.5, ... "label":"Medium" ...}
```

**Prefer clicking? Use Postman:**
1. Method **POST**, URL `http://<PUBLIC_IP>:5000/predict`
2. **Headers:** `Content-Type: application/json`
3. **Body → raw → JSON:** `{"make":"MARUTI","model":"SWIFT VDI","age":3,"km_driven":45000}`
4. **Send** → you should get a `200` with `predicted_price_lakhs`.

> Opening `http://<PUBLIC_IP>:5000/predict` in a **browser** again gives **405** — expected
> (browser = GET). Visiting `http://<PUBLIC_IP>:5000/` (the root) shows friendly usage info.

## 7 · 🧹 Clean up (do this to avoid charges!)

Fargate bills for as long as the task runs. When the test is done, **tear it all down:**

1. **ECS → your cluster → Tasks →** select the task → **Stop** (and **delete the Service** if you made one).
2. **ECS → Clusters →** select `car-price-cluster` → **Delete cluster**.
3. **ECR →** `car-price-api` → select the image → **Delete** (and delete the repo if you like).
4. *(If you created a load balancer or NAT anything, delete those too — they cost the most.)*

> 💸 A single stopped test usually costs a few cents, but **leaving a task RUNNING for days
> adds up.** Deleting is the habit that keeps the AWS free-tier free.

## 8 · Troubleshooting (the usual suspects)

| Symptom | Cause & fix |
| :------ | :---------- |
| `curl: (7) Failed to connect` to the public IP | Security group doesn't allow **inbound TCP 5000**, or **public IP wasn't enabled** on the task. Re-check step 5C. |
| Task shows **STOPPED** seconds after starting | Wrong CPU architecture — rebuild with `docker build --platform linux/amd64 …`, re-push, run again. Check the task's **Logs** tab. |
| `405 Method Not Allowed` in a browser | **Expected.** Browsers send `GET`; `/predict` needs `POST`. Use `curl -X POST` or Postman. |
| `{"status":"unhealthy"}` / `503` | The `models/` folder wasn't in the image. Confirm `COPY models/ ./models/` is in the `Dockerfile` and the folder existed **before** you built. |
| `400 Unknown make/model` | Typo, or a car not in the training data. Casing doesn't matter (it's uppercased), spelling does. |
| ECS can't pull the image | The task definition's **Image URI** is wrong, or ECR is in a **different Region** than the cluster. Keep one Region throughout. |

## 📋 One-page command cheat-sheet

```bash
# LOCAL
docker build -t car-price-api .                 # build the image
docker run -d -p 5000:5000 --name car-price-api car-price-api   # run it
curl http://localhost:5000/health               # test
docker rm -f car-price-api                       # stop + remove

# PUSH TO ECR  (Console: ECR -> repo -> "View push commands" fills these in)
aws ecr get-login-password --region <REGION> | docker login --username AWS --password-stdin <ACCOUNT_ID>.dkr.ecr.<REGION>.amazonaws.com
docker tag car-price-api:latest <ACCOUNT_ID>.dkr.ecr.<REGION>.amazonaws.com/car-price-api:latest
docker push <ACCOUNT_ID>.dkr.ecr.<REGION>.amazonaws.com/car-price-api:latest

# ECS FARGATE  -> easiest via the Console wizard (Part 5). Then:
curl http://<PUBLIC_IP>:5000/health
```

**You did it** 🏁 — a Flask app, packaged in Docker, stored in ECR, and served live on ECS
Fargate. The key mental model to remember: *build once, run anywhere — the container carries
its whole world with it.*